## RAG Pipeline - Data ingestion to Vector DB pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/qz/mw466yp17txddtx0y1qsb1q00000gn/T/ipykernel_35036/4052997885.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
/Users/sanskarvishwakarma/Downloads/projects/AgenticAI/krish_naik_course_agentic_ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdfs inside the dir and convert them into document structure

def load_pdfs_from_directory(directory_path):
    documents = []

    for root, _, files in os.walk(directory_path):
        for file_name in files:
            if file_name.lower().endswith(".pdf"):
                pdf_path = os.path.join(root, file_name)

                loader = PyPDFLoader(pdf_path)
                docs = loader.load()

                for doc in docs:
                    doc.metadata.update({
                        "file_name": file_name,
                        "file_path": pdf_path,
                        "source_directory": root,
                        "document_type": "pdf"
                    })

                documents.extend(docs)

    return documents

In [4]:
all_pdf_documents=load_pdfs_from_directory("./data/pdf")

In [5]:
print(all_pdf_documents)

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-29T15:40:33+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-29T15:40:33+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': "./data/pdf/Sanskar's Resume.pdf", 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': "Sanskar's Resume.pdf", 'file_path': "./data/pdf/Sanskar's Resume.pdf", 'source_directory': './data/pdf', 'document_type': 'pdf'}, page_content='Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — GitHub\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce94%\nTechnical Skills\nProgramming Languages:Java, SQL, JavaScript, TypeScript, Python\

In [6]:
## Chunking the document structure

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better performance"""

    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample Chunks:")
        print(f"Content: {split_docs[0].page_content[:200]}"),
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [7]:
chunks=split_documents(all_pdf_documents,1000,200)
print(chunks)

Split 1 documents into 5 chunks

Example Chunks:
Content: Sanskar Vishwakarma
+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — GitHub
Education
Bachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025
Thakur College of Engineering and Technolog
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-29T15:40:33+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-29T15:40:33+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': "./data/pdf/Sanskar's Resume.pdf", 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': "Sanskar's Resume.pdf", 'file_path': "./data/pdf/Sanskar's Resume.pdf", 'source_directory': './data/pdf', 'document_type': 'pdf'}
[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-29T15:40:33+00:00', 'author': '', 'keywords': ''

### Embeddings and vector DB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """Initialise the embedding manager

        Args:
            model_name: Name of the model used for sentence embeddings
        """

        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts:List[str])->np.ndarray:
        """
        Generate Embeddings for a list of texts

        Args:
            texts: list of strings to embed
        
        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated Embeddings with shape: {embeddings.shape}")

        return embeddings

In [10]:
### initialise the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10320.43it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/qz/mw466yp17txddtx0y1qsb1q00000gn/T/ipykernel_35036/329975365.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDb vector store"""

    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="./data/vector_store"):

        """
        Initialise the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialise_store()

    def _initialise_store(self):
        """Initialise Chroma client and Collection"""

        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialised. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error intialising vector store: {e}")
            raise

    
    def add_documents(self, documents, embeddings):
        """
        Add documents and embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects
            embeddings: List of embedding vectors corresponding to documents
        """

        try:
            ids = []
            texts = []
            metadatas = []

            for i, doc in enumerate(documents):
                ids.append(f"doc_{self.collection.count()}_{i}")
                texts.append(doc.page_content)
                metadatas.append(doc.metadata)

            self.collection.add(
                ids=ids,
                documents=texts,
                embeddings=embeddings,
                metadatas=metadatas
            )

            print(f"Successfully added {len(documents)} documents.")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise

In [12]:
vectorStore=VectorStore()
vectorStore

Vector store initialised. Collection: pdf_documents
Existing documents in collection: 5


In [13]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-29T15:40:33+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-29T15:40:33+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': "./data/pdf/Sanskar's Resume.pdf", 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': "Sanskar's Resume.pdf", 'file_path': "./data/pdf/Sanskar's Resume.pdf", 'source_directory': './data/pdf', 'document_type': 'pdf'}, page_content='Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — GitHub\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce94%\nTechnical Skills\nProgramming Languages:Java, SQL, JavaScript, TypeScript, Python\

In [14]:
## converting chunks into list of strings
texts=[doc.page_content for doc in chunks]
texts

## Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## store embeddings into vector store
vectorStore.add_documents(chunks,embeddings)

Generating embeddings for 5 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.08it/s]

Generated Embeddings with shape: (5, 384)
Successfully added 5 documents.


## Retriever Pipeline from vector store


In [20]:
from langchain_core.documents import Document

class RAGRetriever:
    """Handles query based retrieval from the vector store"""

    def __init__(self,vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialise the retriever

        Args:
            vector_store: Vector store contains document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5) -> List[Document]:
        """
        Retrieve the most relevant documents for a query.

        Args:
            query: User query.
            top_k: Number of documents to retrieve.

        Returns:
            List of retrieved LangChain Document objects.
        """

        if not query.strip():
            return []

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings(query)

        # Search the vector store
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        documents = []

        retrieved_docs = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]

        for doc, metadata, distance in zip(
            retrieved_docs,
            metadatas,
            distances
        ):
            metadata = metadata or {}
            metadata["distance"] = distance

            documents.append(
                Document(
                    page_content=doc,
                    metadata=metadata
                )
            )

        return documents

In [21]:
rag_retriever=RAGRetriever(vector_store=vectorStore,embedding_manager=embedding_manager)

In [22]:
result=rag_retriever.retrieve("What is the name of the person")

Generating embeddings for 30 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Generated Embeddings with shape: (384,)


In [23]:
result

[Document(metadata={'subject': '', 'producer': 'pdfTeX-1.40.27', 'document_type': 'pdf', 'source_directory': './data/pdf', 'trapped': '/False', 'moddate': '2026-06-29T15:40:33+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'keywords': '', 'creator': 'LaTeX with hyperref', 'page_label': '1', 'file_path': "./data/pdf/Sanskar's Resume.pdf", 'page': 0, 'total_pages': 1, 'author': '', 'source': "./data/pdf/Sanskar's Resume.pdf", 'title': '', 'file_name': "Sanskar's Resume.pdf", 'creationdate': '2026-06-29T15:40:33+00:00', 'distance': 1.8558237552642822}, page_content='•Built a full-stack food delivery platform with separate roles for customers, sellers, and riders.\n•Developed restaurant listing, menu management, cart, and Razorpay payment integration features.\n•Implemented nearby restaurant discovery (5 km radius) and real-time rider tracking.\nIntelligent Database Query Agent — Python, Langgraph, Langchain GitHub\n•Bui

### Augmentation with LLM

In [25]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()


True

In [26]:
llm=ChatGroq(model='qwen/qwen3-32b')

In [33]:
def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k)
    context = "\n\n".join(doc.page_content for doc in results)

    if not context:
        return "No Relevant context found for the question"
    
    prompt=f"""
    Use the following context to answer the question concisely, and don't over explain always be at the point

    DONT THINK, GIVE ME DIRECT ANSWER
    
    Context: {context}

    Question: {query}

    Answer:
    """

    response=llm.invoke([
        prompt.format(
            context=context, query=query
        )
    ])

    return response.content

In [34]:
rag_simple("What is the name of the candidate",retriever=rag_retriever,llm=llm)

Generating embeddings for 33 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.97it/s]

Generated Embeddings with shape: (384,)


'<think>\n\n</think>\n\nSanskar Vishwakarma'

In [35]:
rag_simple("How many relevant experience in Java he has ?",retriever=rag_retriever,llm=llm)

Generating embeddings for 45 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.88it/s]


Generated Embeddings with shape: (384,)


'<think>\n\n</think>\n\nSanskar has 2 years of relevant experience in Java.'

In [36]:
rag_simple("What are the skills he has and what makes this resume different from others",retriever=rag_retriever,llm=llm)

Generating embeddings for 75 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.47it/s]

Generated Embeddings with shape: (384,)


'<think>\n\n</think>\n\nSkills: Java, SQL, JavaScript, TypeScript, Python, Spring Boot, REST APIs, React.js, MongoDB, Microservices, System Design, Git, Docker, AI (LangGraph, LangChain, Agentic Workflows).\n\nWhat makes it different: Hands-on experience in both core banking and travel app development with a blend of AI and traditional software engineering skills, showcasing versatility and modern tech integration.'